# 05 Leakage check

Step 12. Runs come from `python -m src.leakage` (logic in `src/leakage.py`), which logs every fold to `results/log.csv`. This notebook only reads the log.

LR and LightGBM on F2 with their Step 9 best configs, scored inside the train block under random stratified 5-fold, StratifiedGroupKFold on uid (5 folds) and the expanding time CV. Random folds let a model see the same customers (and later months) in training, so they are expected to score highest.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import leakage
from src.results import LOG_PATH
from src.data import ROOT

log = pd.read_csv(LOG_PATH)
assert (log['split'] != 'test').all(), 'test block must stay unscored'
best = leakage.best_configs()
table = leakage.leakage_table(log, best)
best

## PR-AUC by model and CV scheme

In [ ]:
pd.set_option('display.precision', 4)
table.pivot(index='model', columns='scheme', values='mean')[list(leakage.SCHEMES)]

In [ ]:
table.pivot(index='model', columns='scheme', values='std')[list(leakage.SCHEMES)]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
schemes = list(leakage.SCHEMES)
width = 0.38
for i, name in enumerate(leakage.MODELS):
    t = table[table.model == name].set_index('scheme').loc[schemes]
    x = np.arange(len(schemes)) + (i - 0.5) * width
    ax.bar(x, t['mean'], width, yerr=t['std'], capsize=3, alpha=0.8, label=leakage.NAMES[name])
    for xi, folds in zip(x, t['folds']):
        ax.scatter([xi] * len(folds), folds, s=10, color='black', zorder=3)
ax.set_xticks(range(len(schemes)), ['random 5-fold', 'group on uid 5-fold', 'time CV (4)'])
ax.set_ylabel('PR-AUC (train-block CV)')
ax.set_title('Leakage check on F2: PR-AUC by CV scheme (dots = folds)')
ax.legend(); fig.tight_layout()
fig.savefig(ROOT / 'reports' / 'figures' / 'leakage_check.png', dpi=150)

## Report text (as written to `reports/leakage_check.md`)

In [ ]:
print(leakage.render(log, best))